In [ ]:
import pandas as pd
import pickle
import scipy.stats as ss
import math
import numpy as np
import datetime
import xgboost as xgb
import os
import sklearn
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRFRegressor 
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score  
from tqdm import tqdm

import warnings
warnings.simplefilter(action='ignore', category=Warning)
pd.options.mode.chained_assignment = None  # default='warn'

In [ ]:
with open("data/raw/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

In [ ]:
data.head()

## Random Forest
### Tuning parameters:
* the depth of trees 𝐿: max_depth
* the number of trees 𝐵 added to the ensemble prediction: n_estimators
* the number of predictors 𝑀 randomly considered as potential split variables: colsample_bylevel

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_val = 1975
start_year_test = 1987
end_year_test = 2016

#Empty containers to save results from each window
predictions = []
y_test_list =[]
dates = []
dic_r2_all = {}

# Model’s complexity: dictionary to save the depth of the trees over time
dic_max_depth_all = {}

# Grid of prespecified values for each hyperparameter  
param_grid = {'max_depth': [2,4,8]}
#convert grid to list containing all posible hyperparameter combinations to iterate over
grid = list(ParameterGrid(param_grid))
# Empty container to save the objective loss function (mean squared error) for each hyperparameter combination
mse = np.full((len(grid),1),np.nan, dtype = np.float32)

In [ ]:
# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1) 
    y_train = train_data['RET']  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1)  
    y_val = val_data['RET'] 
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1)
    y_test = test_data['RET']

    #Loop over the list containing all pyperparameter combinations, fit on the training sample and use 
    #validation set to generate predictions:
    for i in range(len(grid)):
        # random_state: to ensure that results remain stable
        xgbrf_val = XGBRFRegressor(objective='reg:squarederror', random_state=42, learning_rate=1,
                                        max_depth= grid[i]["max_depth"], 
                                        n_estimators = 100,colsample_bylevel=0.3,tree_method='hist')
        xgbrf_val.fit(X_train, y_train)
        Yval_predict = xgbrf_val.predict(X_val)
        #calculate mean squared error for each hyperparameter combination
        mse[i,0] = mean_squared_error(y_val,Yval_predict)


    #The optimal combination of hyperparameters is the one that causes the lowest loss 
    optim_param = grid[np.argmin(mse)]

    #Fit again using the train and validation set and the optimal value for each hyperparameter 
    xgbrf_final = XGBRFRegressor(objective='reg:squarederror', random_state=42,learning_rate=1,
                                max_depth=optim_param["max_depth"],
                                n_estimators = 100,colsample_bylevel=0.2,tree_method='hist')
    xgbrf_final.fit(np.concatenate((X_train, X_val)), np.concatenate((y_train, y_val)))
    preds=xgbrf_final.predict(X_test)


    #Save predictions, dates and the true values of the dependent variable to list
    predictions.append(preds)
    dates.append(y_test.index)
    y_test_list.append(y_test)

    #Calculate OOS model performance the for current window
    r2 = 1-sum(pow(y_test-preds,2))/sum(pow(y_test,2))
    #Save OOS model performance and the respective month to dictionary
    dic_r2_all["r2." + str(y_test.index)] = r2
    # Save the depth of the trees to dictionary to explore how model complexity evolves over time
    dic_max_depth_all["depth." + str(y_test.index)]= optim_param["max_depth"]
   
#Concatenate to get results over the whole OOS test period
predictions_all= np.concatenate(predictions, axis=0)
y_test_list_all= np.concatenate(y_test_list, axis=0) 
dates_all= np.concatenate(dates, axis=0)

R2OOS_RF = 1-sum(pow(y_test_list_all-predictions_all,2))/sum(pow(y_test_list_all,2))
print("R2OOS random forest: ", R2OOS_RF)